# Bronze Layer Bulk Load

This notebook loads raw CSV source files from the **CRM** and **ERP** systems
into the `bronze` schema of a PostgreSQL database (hosted on Supabase).

**Pipeline overview**

1. Connect to the Supabase Postgres database using credentials from a `.env` file.
2. Inspect the tables currently available in the `bronze` schema.
3. Bulk-load each source CSV file into its corresponding bronze table using
   `COPY ... FROM STDIN`, which is the fastest way to bulk-insert data in
   PostgreSQL.
4. Commit each load individually so that a failure in one file does not
   roll back previously loaded tables.

**Source → Target mapping**

| Source file | Target table |
|---|---|
| `datasets/source_crm/cust_info.csv` | `bronze.crm_cust_info` |
| `datasets/source_crm/prd_info.csv` | `bronze.crm_prd_info` |
| `datasets/source_crm/sales_details.csv` | `bronze.crm_sales_details` |
| `datasets/source_erp/cust_az12.csv` | `bronze.erp_cust_az12` |
| `datasets/source_erp/px_cat_g1v2.csv` | `bronze.erp_px_cat_g1v2` |
| `datasets/source_erp/loc_a101.csv` | `bronze.erp_loc_a101` |

**Prerequisites**

- A `.env` file (in the project root) containing:
  ```
  SUPABASE_DATABASE_URL=postgresql://<user>:<password>@<host>:<port>/<db>
  ```
- The `bronze` schema and its target tables must already exist in the database.
- The CSV files must be present at the relative paths shown above.


## 1. Imports and environment setup

Load required libraries and read database credentials from the `.env` file.

In [ ]:
import psycopg2
import os
import re
from urllib.parse import quote
from dotenv import load_dotenv

# Load environment variables from a local .env file
load_dotenv()

## 2. Connect to the database

The connection string is read from the `SUPABASE_DATABASE_URL` environment
variable. Because passwords can contain special characters (e.g. `+`, `@`,
`/`) that are not valid in a URL, the password portion of the connection
string is percent-encoded before connecting.


In [ ]:
def _encode_password_in_dsn(dsn: str) -> str:
    """Percent-encode the password segment of a Postgres DSN.

    This prevents connection errors caused by special characters
    (e.g. '+', '@', '/') that may appear in the database password.
    """
    return re.sub(
        r'://([^:/]+):([^@]+)@',
        lambda m: f'://{m.group(1)}:{quote(m.group(2), safe="")}@',
        dsn,
    )


conn_str = os.getenv("SUPABASE_DATABASE_URL")
if conn_str is None:
    raise ValueError("SUPABASE_DATABASE_URL is not set in the environment")

safe_conn_str = _encode_password_in_dsn(conn_str)

conn = psycopg2.connect(safe_conn_str, sslmode="require")
cur = conn.cursor()

print("Connected to database successfully.")

## 3. Inspect existing bronze tables

Before loading data, list the tables currently present in the `bronze`
schema. This is a sanity check to confirm the target schema and tables
already exist.


In [ ]:
list_tables_query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'bronze'
  AND table_type = 'BASE TABLE'
ORDER BY table_name;
"""

cur.execute(list_tables_query)
tables = cur.fetchall()
tables

In [ ]:
# Roll back to clear the transaction opened by the metadata query above,
# since no data-modifying statements need to be preserved from it.
conn.rollback()

## 4. Load CRM source data

### 4.1 Customer info → `bronze.crm_cust_info`


In [ ]:
with open("../../datasets/source_crm/cust_info.csv", "r") as f:
    cur.copy_expert(
        """
        COPY bronze.crm_cust_info (
            cst_id,
            cst_key,
            cst_firstname,
            cst_lastname,
            cst_material_status,
            cst_gndr,
            cst_create_date
        )
        FROM STDIN
        WITH CSV HEADER
        """,
        f,
    )

conn.commit()
print("Loaded bronze.crm_cust_info")

### 4.2 Product info → `bronze.crm_prd_info`

In [ ]:
with open("../../datasets/source_crm/prd_info.csv", "r", encoding="utf-8") as f:
    cur.copy_expert(
        """
        COPY bronze.crm_prd_info (
            prd_id,
            prd_key,
            prd_nm,
            prd_cost,
            prd_line,
            prd_start_dt,
            prd_end_dt
        )
        FROM STDIN
        WITH (
            FORMAT CSV,
            HEADER TRUE,
            DELIMITER ','
        )
        """,
        f,
    )

conn.commit()
print("Loaded bronze.crm_prd_info")

### 4.3 Sales details → `bronze.crm_sales_details`

In [ ]:
conn.rollback()  # clear any pending transaction state before starting this load

with open("../../datasets/source_crm/sales_details.csv", "r", encoding="utf-8") as f:
    cur.copy_expert(
        """
        COPY bronze.crm_sales_details (
            sls_ord_num,
            sls_prd_key,
            sls_cust_id,
            sls_order_dt,
            sls_ship_dt,
            sls_due_dt,
            sls_sales,
            sls_quantity,
            sls_price
        )
        FROM STDIN
        WITH (
            FORMAT CSV,
            HEADER TRUE,
            DELIMITER ','
        )
        """,
        f,
    )

conn.commit()
print("Loaded bronze.crm_sales_details")

## 5. Load ERP source data

### 5.1 Customer demographics → `bronze.erp_cust_az12`


In [ ]:
conn.rollback()

with open("../../datasets/source_erp/cust_az12.csv", "r", encoding="utf-8") as f:
    cur.copy_expert(
        """
        COPY bronze.erp_cust_az12 (
            cid,
            bdate,
            gen
        )
        FROM STDIN
        WITH (
            FORMAT CSV,
            HEADER TRUE,
            DELIMITER ','
        )
        """,
        f,
    )

conn.commit()
print("Loaded bronze.erp_cust_az12")

### 5.2 Product category reference → `bronze.erp_px_cat_g1v2`

In [ ]:
conn.rollback()

with open("../../datasets/source_erp/px_cat_g1v2.csv", "r", encoding="utf-8") as f:
    cur.copy_expert(
        """
        COPY bronze.erp_px_cat_g1v2 (
            id,
            cat,
            subcat,
            maintenance
        )
        FROM STDIN
        WITH (
            FORMAT CSV,
            HEADER TRUE,
            DELIMITER ','
        )
        """,
        f,
    )

conn.commit()
print("Loaded bronze.erp_px_cat_g1v2")

### 5.3 Customer location → `bronze.erp_loc_a101`

In [ ]:
conn.rollback()

with open("../../datasets/source_erp/loc_a101.csv", "r", encoding="utf-8") as f:
    cur.copy_expert(
        """
        COPY bronze.erp_loc_a101 (
            cid,
            cntry
        )
        FROM STDIN
        WITH (
            FORMAT CSV,
            HEADER TRUE,
            DELIMITER ','
        )
        """,
        f,
    )

conn.commit()
print("Loaded bronze.erp_loc_a101")

## 6. Close the connection

Always close the cursor and connection once all loads are complete to
release database resources.


In [ ]:
cur.close()
conn.close()
print("Database connection closed.")